In [50]:
#!pip install pandas numpy matplotlib missingno

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import missingno as msno

import os
from pathlib import Path

#import sklearn.preprocessing

print("Imported libraries sucessfully! yay")

Imported libraries sucessfully! yay


In [51]:
def find_project_root(
        start: Path = Path.cwd()
        ,project_name: str = "Project_SupervisedLearning_1"
) -> Path:
    """
    Find a project directory containing both data/ and notebooks/ folders.

    Searches:
    1. current dir
    2. current dir / Project_1
    3. parent dirs
    """

    start = start.resolve()

    candidates = [
        start
        ,start / project_name
        ,*start.parents
    ]

    print("Candidates to search: ", candidates)

    checked = set() #unique set to store checked candidates

    for candidate in candidates:
        candidate = candidate.resolve()
        print("Check candidate: ", candidate)
        if candidate in checked:
            print("Already checked! Skip to next candidate")
            continue

        checked.add(candidate)

        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            print("Found ", candidate)
            return candidate

    raise FileNotFoundError(
        f"Could not find {project_name!r} containing data/ and notebooks/."
        "Set PROJECT_ROOT manually using Path()"
    )

In [52]:
PROJECT_ROOT = find_project_root()
print(PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print("Data directory: ", DATA_DIR)

Candidates to search:  [WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/notebooks'), WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/notebooks/Project_SupervisedLearning_1'), WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning'), WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU/DA631E_ArtificialIntelligenceForDataScience'), WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU'), WindowsPath('C:/Users/ngocn/OneDrive/Documents'), WindowsPath('C:/Users/ngocn/OneDrive'), WindowsPath('C:/Users/ngocn'), WindowsPath('C:/Users'), WindowsPath('C:/')]
Check candidate:  C:\Users\ngocn\OneDrive\Documents\MAU\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning\notebooks
Check candidate:  C:\Users\ngocn\OneDrive\Documents\MAU\DA631E_ArtificialIntelligenceForD

In [53]:
def find_unique_file(directory: Path, filename: str) -> Path:
    matches = [path for path in directory.rglob(filename)
                      if path.is_file()]
    print("Matched files: ",matches)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename!r} under {directory}")

    if len(matches) > 1:
        formatted = "\n".join(str(path) for path in matches)
        print("Different formats of dup matches: ", formatted)
        raise RuntimeError(
            f"Found multiple copies of {filename!r}:\n{formatted}"
        )

    return matches[0]

In [54]:
PRICE_PATH = find_unique_file(directory=DATA_DIR, filename="stock_prices.csv")
STOCK_LIST_PATH = find_unique_file(DATA_DIR, "stock_list.csv")
FINANCIALS_PATH = find_unique_file(DATA_DIR, "financials.csv")

print(PRICE_PATH)
print(STOCK_LIST_PATH)
print(FINANCIALS_PATH)

Matched files:  [WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/stock_prices.csv')]
Matched files:  [WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/stock_list.csv')]
Matched files:  [WindowsPath('C:/Users/ngocn/OneDrive/Documents/MAU/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/financials.csv')]
C:\Users\ngocn\OneDrive\Documents\MAU\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning\data\stock_prices.csv
C:\Users\ngocn\OneDrive\Documents\MAU\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning\data\stock_list.csv
C:\Users\ngocn\OneDrive\Documents\MAU\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning\data\financials.csv


In [113]:
prices_raw = pd.read_csv(PRICE_PATH, low_memory=False)
stock_list_raw = pd.read_csv(STOCK_LIST_PATH, low_memory=False)
financials_raw = pd.read_csv(FINANCIALS_PATH, low_memory=False)


In [ ]:
prices = prices_raw.copy()
stock_list = stock_list_raw.copy()
financials_raw = financials_raw.copy()

print("SHAPE")
print("Stock prices: ", prices.shape)
print("stock list (metadata): ", stock_list.shape)
print("financials statements: ", financials.shape)

SHAPE
Stock prices:  (2332531, 12)
stock list (metadata):  (4417, 16)
financials statements:  (92956, 45)


In [57]:
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,NaN,False,0.012324
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,NaN,False,0.006154
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,NaN,False,0.011053
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,NaN,False,0.003026


In [58]:
prices.info()
#date column is in str, need conversion to datetime
#maybe no need for RowId, aldready contained in Date and SecuritiesCode columns

<class 'pandas.DataFrame'>
RangeIndex: 2332531 entries, 0 to 2332530
Data columns (total 12 columns):
 #   Column            Dtype  
---  ------            -----  
 0   RowId             str    
 1   Date              str    
 2   SecuritiesCode    int64  
 3   Open              float64
 4   High              float64
 5   Low               float64
 6   Close             float64
 7   Volume            int64  
 8   AdjustmentFactor  float64
 9   ExpectedDividend  float64
 10  SupervisionFlag   bool   
 11  Target            float64
dtypes: bool(1), float64(7), int64(2), str(2)
memory usage: 198.0 MB


In [59]:
stock_list.head()

,SecuritiesCode,EffectiveDate,Name,Section/Products,NewMarketSegment,33SectorCode,33SectorName,17SectorCode,17SectorName,NewIndexSeriesSizeCode,NewIndexSeriesSize,TradeDate,Close,IssuedShares,MarketCapitalization,Universe0
0,1301,20211230,"KYOKUYO CO.,LTD.",First Section (Domestic),Prime Market,50,"Fishery, Agriculture and Forestry",1,FOODS,7,TOPIX Small 2,20211230.0,3080.0,1.092828e+07,3.365911e+10,True
1,1305,20211230,Daiwa ETF-TOPIX,ETFs/ ETNs,NaN,-,-,-,-,-,-,20211230.0,2097.0,3.634636e+09,7.621831e+12,False
2,1306,20211230,NEXT FUNDS TOPIX Exchange Traded Fund,ETFs/ ETNs,NaN,-,-,-,-,-,-,20211230.0,2073.5,7.917718e+09,1.641739e+13,False
3,1308,20211230,Nikko Exchange Traded Index Fund TOPIX,ETFs/ ETNs,NaN,-,-,-,-,-,-,20211230.0,2053.0,3.736943e+09,7.671945e+12,False
4,1309,20211230,NEXT FUNDS ChinaAMC SSE50 Index Exchange Trade...,ETFs/ ETNs,NaN,-,-,-,-,-,-,20211230.0,44280.0,7.263200e+04,3.216145e+09,False


In [60]:
stock_list.info()

<class 'pandas.DataFrame'>
RangeIndex: 4417 entries, 0 to 4416
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   SecuritiesCode          4417 non-null   int64  
 1   EffectiveDate           4417 non-null   int64  
 2   Name                    4417 non-null   str    
 3   Section/Products        4417 non-null   str    
 4   NewMarketSegment        3772 non-null   str    
 5   33SectorCode            4417 non-null   str    
 6   33SectorName            4417 non-null   str    
 7   17SectorCode            4417 non-null   str    
 8   17SectorName            4417 non-null   str    
 9   NewIndexSeriesSizeCode  4417 non-null   str    
 10  NewIndexSeriesSize      4417 non-null   str    
 11  TradeDate               4121 non-null   float64
 12  Close                   4121 non-null   float64
 13  IssuedShares            4121 non-null   float64
 14  MarketCapitalization    4121 non-null   float64
 15

In [61]:
financials.head()

,DisclosureNumber,DateCode,Date,SecuritiesCode,DisclosedDate,DisclosedTime,DisclosedUnixTime,TypeOfDocument,CurrentPeriodEndDate,TypeOfCurrentPeriod,...,ForecastEarningsPerShare,ApplyingOfSpecificAccountingOfTheQuarterlyFinancialStatements,MaterialChangesInSubsidiaries,ChangesBasedOnRevisionsOfAccountingStandard,ChangesOtherThanOnesBasedOnRevisionsOfAccountingStandard,ChangesInAccountingEstimates,RetrospectiveRestatement,NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock,NumberOfTreasuryStockAtTheEndOfFiscalYear,AverageNumberOfShares
0,2.016121e+13,20170104_2753,2017-01-04,2753.0,2017-01-04,07:30:00,1.483483e+09,3QFinancialStatements_Consolidated_JP,2016-12-31,3Q,...,319.76,NaN,False,True,False,False,False,6848800,－,6848800
1,2.017010e+13,20170104_3353,2017-01-04,3353.0,2017-01-04,15:00:00,1.483510e+09,3QFinancialStatements_Consolidated_JP,2016-11-30,3Q,...,485.36,NaN,False,True,False,False,False,2035000,118917,1916083
2,2.016123e+13,20170104_4575,2017-01-04,4575.0,2017-01-04,12:00:00,1.483499e+09,ForecastRevision,2016-12-31,2Q,...,-93.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.017010e+13,20170105_2659,2017-01-05,2659.0,2017-01-05,15:00:00,1.483596e+09,3QFinancialStatements_Consolidated_JP,2016-11-30,3Q,...,285.05,NaN,False,True,False,False,False,31981654,18257,31963405
4,2.017011e+13,20170105_3050,2017-01-05,3050.0,2017-01-05,15:30:00,1.483598e+09,ForecastRevision,2017-02-28,FY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
financials.info()

<class 'pandas.DataFrame'>
RangeIndex: 92956 entries, 0 to 92955
Data columns (total 45 columns):
 #   Column                                                                        Non-Null Count  Dtype  
---  ------                                                                        --------------  -----  
 0   DisclosureNumber                                                              92954 non-null  float64
 1   DateCode                                                                      92954 non-null  str    
 2   Date                                                                          92956 non-null  str    
 3   SecuritiesCode                                                                92954 non-null  float64
 4   DisclosedDate                                                                 92954 non-null  str    
 5   DisclosedTime                                                                 92954 non-null  str    
 6   DisclosedUnixTime                        

In [63]:
'''
def require_columns(
        dataframe: pd.Da
)
'''


'\ndef require_columns(\n        dataframe: pd.Da\n)\n'

In [64]:
print(prices['SecuritiesCode'].unique().shape)
print(stock_list['SecuritiesCode'].unique().shape)
print(financials['SecuritiesCode'].unique().shape)

print(prices['SecuritiesCode'].dtype)
print(stock_list['SecuritiesCode'].dtype)
print(financials['SecuritiesCode'].dtype)

(2000,)
(4417,)
(4072,)
int64
int64
float64


In [65]:
def normalize_security_code (
        df: pd.DataFrame
        ,dataframe_name: str
) -> pd.DataFrame:
        """
        Validate + Convert SecuritiesCode to nullable Int32 
        (int32 is different from Int32 as int32 does not allow NULL or NA)
        """

        result = df.copy()
        raw_codes = result['SecuritiesCode']

        #Convert numeric strings and existing numbers
        numeric_codes = pd.to_numeric(raw_codes, errors='coerce')

        #Detect non-missing original values that could not be converted
        invalid_format = raw_codes.notna() & numeric_codes.isna()

        if invalid_format.any():
                examples = raw_codes.loc[invalid_format].head()
                raise ValueError(f"{dataframe_name}: invalid SecuritiesCode values: {examples}")

        #A security code must be a whole number
        non_wholenumber = (numeric_codes.notna() & numeric_codes.mod(1).ne(0))

        if non_wholenumber.any():
                examples = numeric_codes.loc[non_wholenumber].head()
                raise ValueError(f"{dataframe_name}: non-whole number SecuritiesCode: {examples}")

        #check that values fits safely inside Int32
        int32_max = np.iinfo(np.int32).max

        outside_range = numeric_codes.notna() & ~numeric_codes.between(0, int32_max)

        if outside_range.any():
                raise ValueError(
                f"{dataframe_name}: SecuritiesCode is outside the Int32 range."
                )

        result["SecuritiesCode"] = numeric_codes.astype("Int32")

        return result
    

In [66]:
prices = normalize_security_code(df=prices, dataframe_name="prices")
stock_list = normalize_security_code(df=stock_list, dataframe_name="stock_list")
financials = normalize_security_code(df=financials, dataframe_name="financials")

print(prices['SecuritiesCode'].unique().shape)
print(stock_list['SecuritiesCode'].unique().shape)
print(financials['SecuritiesCode'].unique().shape)

print(prices['SecuritiesCode'].dtype)
print(stock_list['SecuritiesCode'].dtype)
print(financials['SecuritiesCode'].dtype)

(2000,)
(4417,)
(4072,)
Int32
Int32
Int32


In [67]:
prices.isna().sum()

RowId                     0
Date                      0
SecuritiesCode            0
Open                   7608
High                   7608
Low                    7608
Close                  7608
Volume                    0
AdjustmentFactor          0
ExpectedDividend    2313666
SupervisionFlag           0
Target                  238
dtype: int64

In [68]:
stock_list.isna().sum()

SecuritiesCode              0
EffectiveDate               0
Name                        0
Section/Products            0
NewMarketSegment          645
33SectorCode                0
33SectorName                0
17SectorCode                0
17SectorName                0
NewIndexSeriesSizeCode      0
NewIndexSeriesSize          0
TradeDate                 296
Close                     296
IssuedShares              296
MarketCapitalization      296
Universe0                   0
dtype: int64

In [69]:
financials.isna().sum()

DisclosureNumber                                                                    2
DateCode                                                                            2
Date                                                                                0
SecuritiesCode                                                                      2
DisclosedDate                                                                       2
DisclosedTime                                                                       2
DisclosedUnixTime                                                                   2
TypeOfDocument                                                                      2
CurrentPeriodEndDate                                                                2
TypeOfCurrentPeriod                                                                 2
CurrentFiscalYearStartDate                                                          2
CurrentFiscalYearEndDate                              

In [70]:
financials.loc[financials['SecuritiesCode'].isna()]

,DisclosureNumber,DateCode,Date,SecuritiesCode,DisclosedDate,DisclosedTime,DisclosedUnixTime,TypeOfDocument,CurrentPeriodEndDate,TypeOfCurrentPeriod,...,ForecastEarningsPerShare,ApplyingOfSpecificAccountingOfTheQuarterlyFinancialStatements,MaterialChangesInSubsidiaries,ChangesBasedOnRevisionsOfAccountingStandard,ChangesOtherThanOnesBasedOnRevisionsOfAccountingStandard,ChangesInAccountingEstimates,RetrospectiveRestatement,NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock,NumberOfTreasuryStockAtTheEndOfFiscalYear,AverageNumberOfShares
22557,NaN,NaN,2018-02-21,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54934,NaN,NaN,2019-12-30,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [71]:
financials = financials.dropna(subset=['SecuritiesCode']).copy()

In [72]:
financials.loc[financials['SecuritiesCode'].isna()]

,DisclosureNumber,DateCode,Date,SecuritiesCode,DisclosedDate,DisclosedTime,DisclosedUnixTime,TypeOfDocument,CurrentPeriodEndDate,TypeOfCurrentPeriod,...,ForecastEarningsPerShare,ApplyingOfSpecificAccountingOfTheQuarterlyFinancialStatements,MaterialChangesInSubsidiaries,ChangesBasedOnRevisionsOfAccountingStandard,ChangesOtherThanOnesBasedOnRevisionsOfAccountingStandard,ChangesInAccountingEstimates,RetrospectiveRestatement,NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock,NumberOfTreasuryStockAtTheEndOfFiscalYear,AverageNumberOfShares


In [73]:
financials.info()

<class 'pandas.DataFrame'>
Index: 92954 entries, 0 to 92955
Data columns (total 45 columns):
 #   Column                                                                        Non-Null Count  Dtype  
---  ------                                                                        --------------  -----  
 0   DisclosureNumber                                                              92954 non-null  float64
 1   DateCode                                                                      92954 non-null  str    
 2   Date                                                                          92954 non-null  str    
 3   SecuritiesCode                                                                92954 non-null  Int32  
 4   DisclosedDate                                                                 92954 non-null  str    
 5   DisclosedTime                                                                 92954 non-null  str    
 6   DisclosedUnixTime                             

In [74]:
int(prices.duplicated(['Date']).sum())

2331329

In [75]:
def clean_prices(
        df: pd.DataFrame
) -> tuple[pd.DataFrame, dict]:
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'], errors="raise")

    numeric_cols = ['Open', 'High', 'Low', 'Close','Volume', 'AdjustmentFactor','Target']

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    audit = {
        "input_rows": len(df)
        ,"duplicates_stock_dates": int(df.duplicated(['Date','SecuritiesCode']).sum())
        ,"missing_close": int(df['Close'].isna().sum())
        ,"missing_volume": int(df['Volume'].isna().sum())
        ,"negative_volume": int((df['Volume'] < 0).sum())
             }

    if audit['duplicates_stock_dates']:
        duplicates = df.loc[df.duplicated(['Date','SecuritiesCode'],keep=False)].sort_values(by=['Date','SecuritiesCode'])

        display(duplicates.head(20))
        raise ValueError(
            "Duplicate Date-SecuritiesCode records found. "
            "Investigate before continuing"
        )

    complete_ohlc = df[['Open', 'High', 'Low', 'Close']].notna().all(axis=1)
    invalid_ohlc = complete_ohlc & (
        (df['High'] < df['Low'])
        | (df['High'] < df['Open'])
        | (df['High'] < df['Close'])
        | (df['Low'] > df['Open'])
        | (df['Low'] > df['Close'])
        | (df['Close'] <= 0)
        | (df['Open'] <= 0)
    )

    audit['invalid_ohcl'] = int(invalid_ohlc.sum())

    if invalid_ohlc.any():
        print(
            f"Removing {invalid_ohlc.sum():,} rows with internally inconsistent OHLC values.")

        df = df.loc[~invalid_ohlc].copy()

    invalid_volume = df['Volume'].isna() | df['Volume'] <0
    audit['invalid_volume_rows'] = int(invalid_volume.sum())

    df = df.loc[~invalid_volume].copy()

    missing_adjustment = df['AdjustmentFactor'].isna()
    audit['missing_adjustment'] = int(missing_adjustment.sum())

    print("Fill missing adjustment with 1.0 (unchanged stock split)")
    df.loc[missing_adjustment, 'AdjustmentFactor'] = 1.0

    if (df['AdjustmentFactor'] <= 0).any():
        raise ValueError("AdjustmentFactor contains non-positive values.")

    df = df.sort_values(['SecuritiesCode','Date']).reset_index(drop=True)

    audit['output_rows'] = len(df)
    audit['removed_rows'] = audit['output_rows'] - audit['input_rows']

    return df, audit

In [76]:
prices, prices_audit = clean_prices(df=prices)
pd.Series(prices_audit, name="value")

Fill missing adjustment with 1.0 (unchanged stock split)


input_rows                2332531
duplicates_stock_dates          0
missing_close                7608
missing_volume                  0
negative_volume                 0
invalid_ohcl                    0
invalid_volume_rows             0
missing_adjustment              0
output_rows               2332531
removed_rows                    0
Name: value, dtype: int64

In [77]:
prices[prices['Close'].isna()].sort_values(by='Date',ascending=False)

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
1067969,20211203_5918,2021-12-03,5918,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.015625
305621,20211203_2761,2021-12-03,2761,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000
2102717,20211203_9083,2021-12-03,9083,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.009615
38107,20211203_1787,2021-12-03,1787,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.030351
2250794,20211203_9733,2021-12-03,9733,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
493438,20170106_3540,2017-01-06,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
821244,20170105_4621,2017-01-05,4621,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000
493437,20170105_3540,2017-01-05,3540,NaN,NaN,NaN,NaN,0,1.0,NaN,False,NaN
2197907,20170104_9539,2017-01-04,9539,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.004149


In [84]:
ohlc =  ['Open', 'High', 'Low', 'Close']
missing_close = prices['Close'].isna()
missing_close_audit = pd.Series({
    'missing_close_rows': int(missing_close.sum())
    ,'all_ohlc_missing': int(prices.loc[missing_close, ohlc].isna().all(axis=1).sum())
    ,'volume_zero': int(prices.loc[missing_close,'Volume'].eq(0).sum())
    ,'target_available': int(prices.loc[missing_close,'Target'].notna().sum())
    ,'target_missing': int(prices.loc[missing_close,'Target'].isna().sum())
    ,'affected_stocks': int(prices.loc[missing_close, 'SecuritiesCode'].nunique())
})
missing_close_audit

missing_close_rows    7608
all_ohlc_missing      7608
volume_zero           7608
target_available      7370
target_missing         238
affected_stocks       1992
dtype: int64

In [86]:
prices.loc[missing_close].groupby('Date').size().sort_values(ascending=False).head(20)

Date
2020-10-01    1988
2017-03-16      15
2019-04-04      14
2019-10-09      14
2021-10-29      13
2018-03-07      12
2017-03-30      12
2020-07-16      12
2020-09-17      12
2020-05-14      12
2017-02-07      12
2021-08-05      12
2017-01-20      12
2021-11-11      12
2018-08-23      12
2018-07-11      11
2019-03-28      11
2020-10-19      11
2018-09-06      11
2018-08-16      11
dtype: int64

In [93]:
daily_status = (
    prices.groupby("Date")
    .agg(total_rows=("SecuritiesCode", "size"),missing_close_rows=("Close",lambda values: values.isna().sum())
    ,target_available=("Target","count"))
)

daily_status["missing_close_rate"] = (
    daily_status["missing_close_rows"]
    / daily_status["total_rows"]
)

daily_status.sort_values(by='missing_close_rate', ascending=False)

,total_rows,missing_close_rows,target_available,missing_close_rate
Date,,,,
2020-10-01,1988,1988,1988,1.0
2017-03-16,1867,15,1866,0.008034
2019-04-04,1938,14,1938,0.007224
2019-10-09,1951,14,1951,0.007176
2021-10-29,2000,13,2000,0.0065
...,...,...,...,...
2021-05-06,2000,0,2000,0.0
2020-03-24,1971,0,1971,0.0
2020-03-19,1971,0,1971,0.0


In [95]:
all_ohlc_missing = prices[ohlc].isna().all(axis=1)
prices['NoTradeFlag'] = (all_ohlc_missing & prices['Volume'].eq(0)).astype("int8")
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730,0
1,20170105_1301,2017-01-05,1301,2743.0,2747.0,2735.0,2738.0,17900,1.0,NaN,False,0.002920,0
2,20170106_1301,2017-01-06,1301,2734.0,2744.0,2720.0,2740.0,19900,1.0,NaN,False,-0.001092,0
3,20170110_1301,2017-01-10,1301,2745.0,2754.0,2735.0,2748.0,24200,1.0,NaN,False,-0.005100,0
4,20170111_1301,2017-01-11,1301,2748.0,2752.0,2737.0,2745.0,9300,1.0,NaN,False,-0.003295,0


In [96]:
daily_market_status = (
    prices.groupby("Date")
    .agg(total_rows=("SecuritiesCode", "size")
    ,no_trade_securitiescode =('NoTradeFlag','sum')
    ,target_available=("Target","count"))
)

daily_market_status["no_trade_rate"] = (
    daily_market_status["no_trade_securitiescode"]
    / daily_status["total_rows"]
)

daily_market_status.sort_values(by='no_trade_rate', ascending=False)

,total_rows,no_trade_securitiescode,target_available,no_trade_rate
Date,,,,
2020-10-01,1988,1988,1988,1.0
2017-03-16,1867,15,1866,0.008034
2019-04-04,1938,14,1938,0.007224
2019-10-09,1951,14,1951,0.007176
2021-10-29,2000,13,2000,0.0065
...,...,...,...,...
2021-05-06,2000,0,2000,0.0
2020-03-24,1971,0,1971,0.0
2020-03-19,1971,0,1971,0.0


In [99]:
daily_market_status['MarketWithNoTradeFlag'] = daily_market_status['no_trade_rate'] >= 0.9
market_status_by_date = daily_market_status['MarketWithNoTradeFlag']
market_status_by_date

Date
2017-01-04    False
2017-01-05    False
2017-01-06    False
2017-01-10    False
2017-01-11    False
              ...  
2021-11-29    False
2021-11-30    False
2021-12-01    False
2021-12-02    False
2021-12-03    False
Name: MarketWithNoTradeFlag, Length: 1202, dtype: boolean

In [104]:
market_status_by_date['2020-10-01']

np.True_

In [100]:
prices['MarketWithNoTradeFlag'] = prices['Date'].map(market_status_by_date).fillna(False).astype("int8")
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730,0,0
1,20170105_1301,2017-01-05,1301,2743.0,2747.0,2735.0,2738.0,17900,1.0,NaN,False,0.002920,0,0
2,20170106_1301,2017-01-06,1301,2734.0,2744.0,2720.0,2740.0,19900,1.0,NaN,False,-0.001092,0,0
3,20170110_1301,2017-01-10,1301,2745.0,2754.0,2735.0,2748.0,24200,1.0,NaN,False,-0.005100,0,0
4,20170111_1301,2017-01-11,1301,2748.0,2752.0,2737.0,2745.0,9300,1.0,NaN,False,-0.003295,0,0


In [105]:
prices[prices['Date'] == '2020-10-01']

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag
913,20201001_1301,2020-10-01,1301,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.029208,1,1
2115,20201001_1332,2020-10-01,1332,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.027211,1,1
3317,20201001_1333,2020-10-01,1333,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.027695,1,1
3614,20201001_1375,2020-10-01,1375,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.023833,1,1
4816,20201001_1376,2020-10-01,1376,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.022152,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2327434,20201001_9990,2020-10-01,9990,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.023297,1,1
2328636,20201001_9991,2020-10-01,9991,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.041621,1,1
2329838,20201001_9993,2020-10-01,9993,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.034006,1,1
2331040,20201001_9994,2020-10-01,9994,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.025047,1,1


In [107]:
prices['StockSpecificNoTradeFlag'] = (prices['NoTradeFlag'].eq(1) & prices['MarketWithNoTradeFlag'].eq(0)).astype('int8')
prices[prices['StockSpecificNoTradeFlag']>0]

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag,StockSpecificNoTradeFlag
7728,20171121_1381,2017-11-21,1381,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.013514,1,0,1
8372,20200716_1381,2020-07-16,1381,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.004728,1,0,1
30155,20181114_1723,2018-11-14,1723,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.014123,1,0,1
30209,20190207_1723,2019-02-07,1723,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.006531,1,0,1
30224,20190301_1723,2019-03-01,1723,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.002246,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2318951,20200619_9977,2020-06-19,9977,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.001912,1,0,1
2319249,20210907_9977,2021-09-07,9977,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.003333,1,0,1
2319293,20211111_9977,2021-11-11,9977,NaN,NaN,NaN,NaN,0,1.0,NaN,False,-0.009689,1,0,1
2319297,20211117_9977,2021-11-17,9977,NaN,NaN,NaN,NaN,0,1.0,NaN,False,0.000000,1,0,1


In [109]:
prices[
    [
        "NoTradeFlag",
        "MarketWithNoTradeFlag",
        "StockSpecificNoTradeFlag",
    ]
].sum()

NoTradeFlag                 7608
MarketWithNoTradeFlag       1988
StockSpecificNoTradeFlag    5620
dtype: int64

In [ ]:
prices.corr()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag,StockSpecificNoTradeFlag
RowId,1.000000,0.984664,-0.002366,0.027686,0.028284,0.026852,0.027525,-0.020065,-0.010067,0.059735,0.003972,-0.002972,0.011332,0.021572,0.000339
Date,0.984664,1.000000,-0.002450,0.030181,0.030733,0.029383,0.030009,-0.021693,-0.008811,0.043939,0.004641,-0.001507,0.013411,0.025766,0.000260
SecuritiesCode,-0.002366,-0.002450,1.000000,0.016788,0.016348,0.017235,0.016798,0.040869,0.001716,0.033389,-0.001591,-0.003389,-0.005367,-0.000129,-0.006165
Open,0.027686,0.030181,0.016788,1.000000,0.999851,0.999854,0.999730,-0.036524,-0.006918,0.558148,-0.004112,-0.003919,NaN,NaN,NaN
High,0.028284,0.030733,0.016348,0.999851,1.000000,0.999756,0.999857,-0.036316,-0.006933,0.557004,-0.004114,-0.003809,NaN,NaN,NaN
Low,0.026852,0.029383,0.017235,0.999854,0.999756,1.000000,0.999862,-0.036639,-0.006893,0.558808,-0.004092,-0.003958,NaN,NaN,NaN
Close,0.027525,0.030009,0.016798,0.999730,0.999857,0.999862,1.000000,-0.036473,-0.006907,0.558037,-0.004101,-0.003852,NaN,NaN,NaN
Volume,-0.020065,-0.021693,0.040869,-0.036524,-0.036316,-0.036639,-0.036473,1.000000,0.005304,-0.019052,0.101938,-0.000873,-0.010120,-0.005167,-0.008694
AdjustmentFactor,-0.010067,-0.008811,0.001716,-0.006918,-0.006933,-0.006893,-0.006907,0.005304,1.000000,NaN,-0.000113,-0.000091,-0.000073,-0.000219,0.000045
ExpectedDividend,0.059735,0.043939,0.033389,0.558148,0.557004,0.558808,0.558037,-0.019052,NaN,1.000000,-0.006183,-0.148628,0.003139,NaN,0.003139


In [112]:
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag,StockSpecificNoTradeFlag
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730,0,0,0
1,20170105_1301,2017-01-05,1301,2743.0,2747.0,2735.0,2738.0,17900,1.0,NaN,False,0.002920,0,0,0
2,20170106_1301,2017-01-06,1301,2734.0,2744.0,2720.0,2740.0,19900,1.0,NaN,False,-0.001092,0,0,0
3,20170110_1301,2017-01-10,1301,2745.0,2754.0,2735.0,2748.0,24200,1.0,NaN,False,-0.005100,0,0,0
4,20170111_1301,2017-01-11,1301,2748.0,2752.0,2737.0,2745.0,9300,1.0,NaN,False,-0.003295,0,0,0


In [ ]:
prices.isna().sum()

RowId                             0
Date                              0
SecuritiesCode                    0
Open                           7608
High                           7608
Low                            7608
Close                          7608
Volume                            0
AdjustmentFactor                  0
ExpectedDividend            2313666
SupervisionFlag                   0
Target                          238
NoTradeFlag                       0
MarketWithNoTradeFlag             0
StockSpecificNoTradeFlag          0
dtype: int64

In [118]:
n_before =len(prices)
print("Length before exlcuding market with no trade: ", n_before)

prices = prices.loc[prices['MarketWithNoTradeFlag'].eq(0)].copy()
print(f"Length after: {len(prices)},{len(prices)-n_before} obs")

Length before exlcuding market with no trade:  2332531
Length after: 2330543,-1988 obs


In [119]:
prices.isna().sum()

RowId                             0
Date                              0
SecuritiesCode                    0
Open                           5620
High                           5620
Low                            5620
Close                          5620
Volume                            0
AdjustmentFactor                  0
ExpectedDividend            2311678
SupervisionFlag                   0
Target                          238
NoTradeFlag                       0
MarketWithNoTradeFlag             0
StockSpecificNoTradeFlag          0
dtype: int64

In [120]:
prices['HasExpectedDividend'] = prices['ExpectedDividend'].notna().astype('int8')
prices[prices['HasExpectedDividend']==1]

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend
56,20170327_1301,2017-03-27,1301,3125.0,3135.0,3110.0,3120.0,89300,1.0,60.0,False,-0.033708,0,0,0,1
301,20180326_1301,2018-03-26,1301,3785.0,3860.0,3780.0,3860.0,43900,1.0,60.0,False,-0.033079,0,0,0,1
545,20190325_1301,2019-03-25,1301,3015.0,3040.0,2995.0,3040.0,51300,1.0,60.0,False,-0.020667,0,0,0,1
787,20200326_1301,2020-03-26,1301,2650.0,2714.0,2601.0,2689.0,65000,1.0,70.0,False,-0.050054,0,0,0,1
1032,20210326_1301,2021-03-26,1301,3250.0,3285.0,3245.0,3255.0,48700,1.0,70.0,False,-0.038941,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2331996,20190925_9997,2019-09-25,9997,712.0,720.0,705.0,709.0,298300,1.0,8.0,False,-0.034868,0,0,0,1
2332116,20200326_9997,2020-03-26,9997,489.0,489.0,468.0,485.0,262600,1.0,8.0,False,-0.064151,0,0,0,1
2332238,20200925_9997,2020-09-25,9997,934.0,964.0,934.0,959.0,388500,1.0,8.0,False,0.029744,0,0,0,1
2332361,20210326_9997,2021-03-26,9997,1312.0,1333.0,1306.0,1317.0,174700,1.0,8.5,False,0.000762,0,0,0,1


In [121]:
prices['ExpectedDividend'] = prices['ExpectedDividend'].fillna(0.0)

In [122]:
prices.isna().sum()

RowId                          0
Date                           0
SecuritiesCode                 0
Open                        5620
High                        5620
Low                         5620
Close                       5620
Volume                         0
AdjustmentFactor               0
ExpectedDividend               0
SupervisionFlag                0
Target                       238
NoTradeFlag                    0
MarketWithNoTradeFlag          0
StockSpecificNoTradeFlag       0
HasExpectedDividend            0
dtype: int64

In [124]:
prices = prices.sort_values(['SecuritiesCode','Date']).reset_index(drop=True)
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,False,0.000730,0,0,0,0
1,20170105_1301,2017-01-05,1301,2743.0,2747.0,2735.0,2738.0,17900,1.0,0.0,False,0.002920,0,0,0,0
2,20170106_1301,2017-01-06,1301,2734.0,2744.0,2720.0,2740.0,19900,1.0,0.0,False,-0.001092,0,0,0,0
3,20170110_1301,2017-01-10,1301,2745.0,2754.0,2735.0,2748.0,24200,1.0,0.0,False,-0.005100,0,0,0,0
4,20170111_1301,2017-01-11,1301,2748.0,2752.0,2737.0,2745.0,9300,1.0,0.0,False,-0.003295,0,0,0,0


In [126]:
prices['LastTradeDate'] = prices['Date'].where(prices['Close'].notna())

prices['LastTradeDate'] = prices.groupby(by='SecuritiesCode',sort=False)['LastTradeDate'].ffill()

prices['DaysSinceLastTrade'] = (prices['Date'] - prices['LastTradeDate']).dt.days
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,NoTradeFlag,MarketWithNoTradeFlag,StockSpecificNoTradeFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,False,0.000730,0,0,0,0,2017-01-04,0.0
1,20170105_1301,2017-01-05,1301,2743.0,2747.0,2735.0,2738.0,17900,1.0,0.0,False,0.002920,0,0,0,0,2017-01-05,0.0
2,20170106_1301,2017-01-06,1301,2734.0,2744.0,2720.0,2740.0,19900,1.0,0.0,False,-0.001092,0,0,0,0,2017-01-06,0.0
3,20170110_1301,2017-01-10,1301,2745.0,2754.0,2735.0,2748.0,24200,1.0,0.0,False,-0.005100,0,0,0,0,2017-01-10,0.0
4,20170111_1301,2017-01-11,1301,2748.0,2752.0,2737.0,2745.0,9300,1.0,0.0,False,-0.003295,0,0,0,0,2017-01-11,0.0


In [127]:
prices['CumulativeAdjustment'] = prices.groupby(by='SecuritiesCode')[
    'AdjustmentFactor'].transform(lambda values: (values.fillna(1.0).iloc[::-1].cumprod().iloc[::-1]))

prices['AdjustedClose'] = prices['Close'] * prices['CumulativeAdjustment']

In [128]:
grouped_close = prices.groupby(by='SecuritiesCode')['AdjustedClose']
prices['CloseLag1'] = grouped_close.shift(periods=1)
prices['CloseLag5'] = grouped_close.shift(periods=5)
prices['CloseLag20'] = grouped_close.shift(periods=20)
prices['CloseLag60'] = grouped_close.shift(periods=60)
prices.head()

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,...,StockSpecificNoTradeFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade,CumulativeAdjustment,AdjustedClose,CloseLag1,CloseLag5,CloseLag20,CloseLag60
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,0.0,...,0,0,2017-01-04,0.0,1.0,2742.0,NaN,NaN,NaN,NaN
1,20170105_1301,2017-01-05,1301,2743.0,2747.0,2735.0,2738.0,17900,1.0,0.0,...,0,0,2017-01-05,0.0,1.0,2738.0,2742.0,NaN,NaN,NaN
2,20170106_1301,2017-01-06,1301,2734.0,2744.0,2720.0,2740.0,19900,1.0,0.0,...,0,0,2017-01-06,0.0,1.0,2740.0,2738.0,NaN,NaN,NaN
3,20170110_1301,2017-01-10,1301,2745.0,2754.0,2735.0,2748.0,24200,1.0,0.0,...,0,0,2017-01-10,0.0,1.0,2748.0,2740.0,NaN,NaN,NaN
4,20170111_1301,2017-01-11,1301,2748.0,2752.0,2737.0,2745.0,9300,1.0,0.0,...,0,0,2017-01-11,0.0,1.0,2745.0,2748.0,NaN,NaN,NaN


In [129]:
prices['LogReturn1Day'] = np.log(prices['AdjustedClose'] / prices['CloseLag1'])

prices['Momentum5Days'] = prices['AdjustedClose'] / prices['CloseLag5'] - 1
prices['Momentum20Days'] = prices['AdjustedClose'] / prices['CloseLag20'] - 1
prices['Momentum60Days'] = prices['AdjustedClose'] / prices['CloseLag60'] - 1

prices['MovingAverage20Days'] = grouped_close.transform(lambda values: values.rolling(window=20, min_periods=15).mean())
prices['ClosetoMovingAverage20Days'] = prices['AdjustedClose'] / prices['MovingAverage20Days'] - 1 

In [80]:
META_CANDIDATES = [
    'SecuritiesCode'
    ,'33SectorCode'
    ,'17SectorCode'
    ,'NewMarketSegment'
]

meta_columns = [column for column in META_CANDIDATES if column in stock_list.columns]
stock_meta = stock_list[meta_columns].copy()

duplicate_meta = stock_meta.duplicated('SecuritiesCode').copy()
if duplicate_meta.any():
    display(stock_meta.loc[duplicate_meta].sort_values('SecuritiesCode').head(20))
    raise ValueError("stock_list has multiple rows per security code")

for column in meta_columns:
    if column != 'SecuritiesCode':
        stock_meta[column] = stock_meta[column].astype("string").fillna("__MISSING__")